# 2. Serving endpoint path

This notebook deploys a Docling model serving endpoint and shows the agent tool call pattern.

What it does:
- Registers a single-file Docling parsing model in Unity Catalog
- Creates or updates a serving endpoint
- Calls the endpoint by passing input paths and receiving output paths

In [ ]:
%pip install uv
%sh uv pip install .
%restart_python

In [22]:
import mlflow
import tomllib

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

config = mlflow.models.ModelConfig(development_config="./config.yaml")
config = config.to_dict()

with open("pyproject.toml", "rb") as handle:
    pyproject = tomllib.load(handle)

optional_deps = pyproject.get("project", {}).get("optional-dependencies", {})
serving_deps = optional_deps.get("serving") or optional_deps.get("deployment") or []

REGISTERED_MODEL_NAME = f"{config['catalog']}.{config['schema']}.{config['model_name']}"
ENDPOINT_NAME = config["serving_endpoint"]

In [23]:
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment('/Workspace/Users/scott.mckean@databricks.com/experiments/docling')

simple_paths = [
  '/Volumes/shm/default/raw_pdfs/cauli_wingz.pdf'
]


In [ ]:
w = WorkspaceClient()
w.secrets.get_secret(scope="shm", key="sp-id").value

'MzgzY2Y4YzQtN2ZiNy00Y2Y4LWFkYjktMWFjOGQxYWYxZGUx'

In [50]:
import base64

def get_decoded_secret(scope, key):
    encoded = w.secrets.get_secret(scope=scope, key=key).value
    return base64.b64decode(encoded).decode("utf-8")

In [55]:
import os
w = WorkspaceClient(profile="default")
w.secrets.get_secret(scope="shm", key="sp-id")
w.secrets.get_secret(scope="shm", key="sp-auth")

os.environ["DATABRICKS_HOST"] = "https://adb-984752964297111.11.azuredatabricks.net/"
os.environ["DATABRICKS_CLIENT_ID"] = get_decoded_secret(scope=config["secret_scope_name"], key=config["client_id_key"])
os.environ["DATABRICKS_CLIENT_SECRET"] = get_decoded_secret(scope=config["secret_scope_name"], key=config["client_secret_key"])

In [54]:
w = WorkspaceClient(profile="default")

with mlflow.start_run(run_name="docling_endpoint_registration"):
    model_info = mlflow.pyfunc.log_model(
        name="docling_parser",
        python_model='./docling_endpoint.py',
        input_example=simple_paths,
        pip_requirements=serving_deps,
        registered_model_name=REGISTERED_MODEL_NAME,
    )

ValueError: invalid_client: Client authentication failed

In [ ]:
from tkinter import W


w = WorkspaceClient()

served_entity = ServedEntityInput(
    name="docling-parser",
    entity_name=REGISTERED_MODEL_NAME,
    entity_version=model_info.registered_model_version,
    workload_size="Small",
    workload_type="GPU_SMALL",
    scale_to_zero_enabled=True,
)

try:
    W.serving_endpoints.get(ENDPOINT_NAME)
    w.serving_endpoints.update_config_and_wait(
        name=ENDPOINT_NAME,
        served_entities=[served_entity],
    )
except Exception:
    w.serving_endpoints.create_and_wait(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(served_entities=[served_entity]),
    )

AttributeError: 'ModelInfo' object has no attribute 'registered_model'

In [ ]:



model_uri = model_info.model_uri
registered_model = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)
ensure_endpoint(MODEL_NAME, registered_model.version, ENDPOINT_NAME)
print(f"Serving endpoint ready: {ENDPOINT_NAME}")

In [ ]:
from mlflow.deployments import get_deploy_client

sample_inputs = config.get(
    "evaluate_sample_inputs",
    [f"/Volumes/{config['catalog']}/{config['schema']}/{config['input_volume']}/sample.pdf"],
)
output_root = f"/Volumes/{config['catalog']}/{config['schema']}/{config['output_volume']}"
options = {"generate_page_images": False}

rows = [[path, output_root, options] for path in sample_inputs]
request = {
    "dataframe_split": {
        "columns": ["file_path", "output_root", "options"],
        "data": rows,
    }
}

deploy_client = get_deploy_client("databricks")
response = deploy_client.predict(endpoint=ENDPOINT_NAME, inputs=request)
predictions = response.get("predictions", response)

output_paths = [
    item["output_path"]
    for item in predictions
    if item.get("status") == "success" and item.get("output_path")
]

print("Parsed output paths:")
for path in output_paths:
    print(path)